# Playing with semantic retrieval

The goal here is:

Can we take a complaint, convert it into an embedding, and retrieve other complaints discussing the same underlying issue?

I am not interested in building a chatbot. The objective is to understand what these embeddings are actually capturing and whether they can help identify recurring patterns in complaint data.

In [ ]:
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import umap
import plotly.express as px

## Generate embeddings

This is the interesting part.

Each complaint narrative is converted into a vector representation.

The idea is that complaints discussing similar issues should end up close together in vector space, even if they use different words.

Coming from a physics background, I find it useful to think of this as moving from language into geometry.

In [ ]:

df_sample = pd.read_parquet("../data/processed/complaints_50k.parquet")

texts = df_sample[
    "Consumer complaint narrative"
].tolist()

model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

## Generate embeddings for the texts
texts = (
    df_sample["Consumer complaint narrative"]
    .dropna()
    .astype(str)
    .tolist()
)
sample_texts = texts[:2000]
embeddings = model.encode(
    sample_texts,
    show_progress_bar=True
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\pablo\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\pablo\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

## Retrieve similar complaints

Let's start with a single complaint and see what the model thinks is similar.

At this stage I am not trying to classify anything. I just want to understand whether semantic retrieval actually works.

In [3]:


# Pick a complaint to investigate
query_index = 0
query = sample_texts[query_index]

# Compare it against all complaints
query_embedding = embeddings[query_index].reshape(1, -1)

similarities = cosine_similarity(
    query_embedding,
    embeddings
)

# Retrieve the 20 most similar complaints
top_indices = similarities[0].argsort()[-20:][::-1]

print("Query complaint:\n")
print(query[:1000])

print("\nMost similar complaints:\n")

for idx in top_indices[1:6]:
    print("\n" + "=" * 80)
    print(f"Similarity: {similarities[0][idx]:.3f}")
    print(sample_texts[idx][:500])

Query complaint:

Equifax is reporting an original creditor account XXXX XXXX with account number xxxxxxxxxxxx XXXX on my report that was charged off and sold to a collection agency. I settled the debt with the collection agency in full. The original creditor already charged off the account on their taxes and made money off selling the debt to the collection agency I paid, but they are still reporting it on my credit report, which brings down my score and affect my credit. I have already cleared up the debt and Equifax is still allowing the original creditor to report the debt which violates my Fair Credit Reporting rights.

Most similar complaints:


Similarity: 0.716
The collection agency placed a bad debt on my credit report for XXXX when the balance was paid directly to the creditor. I do not owe Credence collection agency and this is a false representation that is being displayed on my XXXX  report. This needs to be removed completely from my account.

Similarity: 0.710
Equfax has

## Project the embedding space into 2D

The embeddings live in a 384-dimensional space.

That's fine for computation but impossible to visualize.

UMAP gives us a way to project that space into two dimensions and inspect the local structure.

The projection is not perfect, but it should preserve neighborhoods reasonably well. It is cool to see, but at the same time we make peace with this 2D projection being some sort of homeopathy for engineers. You would love to see something but it´s unlikely.

In [4]:
# Projection

reducer = umap.UMAP(
    n_neighbors=15,
    min_dist=0.1,
    metric="cosine",
    random_state=42
)

embedding_2d = reducer.fit_transform(embeddings)

c:\Users\pablo\anaconda3\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


### Visualizing the neighborhood

Each point represents a complaint.

Gray points are all complaints in the sample.

The red point is the complaint being investigated.

Blue points are the complaints retrieved through cosine similarity.

If the embeddings are meaningful, the blue points should end up near the red point.

In [5]:
import pandas as pd
import plotly.graph_objects as go

plot_df = pd.DataFrame({
    "x": embedding_2d[:, 0],
    "y": embedding_2d[:, 1]
})

plot_df["group"] = "Other"

plot_df.loc[top_indices, "group"] = "Similar"
plot_df.loc[query_index, "group"] = "Query"

fig = go.Figure()

# Gray background cloud
others = plot_df[plot_df["group"] == "Other"]

fig.add_trace(
    go.Scattergl(
        x=others["x"],
        y=others["y"],
        mode="markers",
        name="Other complaints",
        marker=dict(
            size=4,
            color="lightgray"
        ),
        opacity=0.3
    )
)

# Blue semantic neighbors
similar = plot_df[plot_df["group"] == "Similar"]

fig.add_trace(
    go.Scattergl(
        x=similar["x"],
        y=similar["y"],
        mode="markers",
        name="Semantic neighbors",
        marker=dict(
            size=12,
            color="blue"
        )
    )
)

# Big red query point
query = plot_df[plot_df["group"] == "Query"]

fig.add_trace(
    go.Scattergl(
        x=query["x"],
        y=query["y"],
        mode="markers",
        name="Query complaint",
        marker=dict(
            size=20,
            color="red"
        )
    )
)

fig.update_layout(
    title="Semantic Neighborhood Around Selected Complaint",
    width=1000,
    height=800,
    template="plotly_white"
)

fig.show()

In [6]:
for idx in top_indices:

    print("\n" + "=" * 100)
    print(f"INDEX: {idx}")
    print(f"SIMILARITY: {similarities[0][idx]:.3f}")
    print()

    print(sample_texts[idx][:1000])


INDEX: 0
SIMILARITY: 1.000

Equifax is reporting an original creditor account XXXX XXXX with account number xxxxxxxxxxxx XXXX on my report that was charged off and sold to a collection agency. I settled the debt with the collection agency in full. The original creditor already charged off the account on their taxes and made money off selling the debt to the collection agency I paid, but they are still reporting it on my credit report, which brings down my score and affect my credit. I have already cleared up the debt and Equifax is still allowing the original creditor to report the debt which violates my Fair Credit Reporting rights.

INDEX: 629
SIMILARITY: 0.716

The collection agency placed a bad debt on my credit report for XXXX when the balance was paid directly to the creditor. I do not owe Credence collection agency and this is a false representation that is being displayed on my XXXX  report. This needs to be removed completely from my account.

INDEX: 44
SIMILARITY: 0.710

Equ

## First impressions

The retrieval results are surprisingly good.

The model is not simply matching keywords. It retrieves complaints that appear to discuss the same underlying issue even when the wording differs significantly.

In the example explored here, the retrieved complaints consistently revolve around:

- Credit reporting disputes
- Settled or paid debts
- Collection agencies
- Incorrect reporting by credit bureaus

This feels like the beginning of an investigation workflow rather than a search engine.

Next step:

Instead of looking at individual neighbors, build a semantic graph and explore how larger complaint communities emerge from these local relationships.